# Adaptive LoRA Router

Notebook 01

Environment Setup

Goal:

• Verify Colab GPU
• Install dependencies
• Load Qwen 3B
• Verify inference
• Measure GPU memory

Expected Output:

A working Qwen model ready for LoRA fine tuning.

### Step 1
Verify colab T4 gpu

In [2]:
import torch
print("Is CUDA available?", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Is CUDA available? True
Device Name: Tesla T4


Installing unsloth dependencies

In [4]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-evouqbl0/unsloth_77263bd1d5ae4599b37f001b17983868
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-evouqbl0/unsloth_77263bd1d5ae4599b37f001b17983868
  Resolved https://github.com/unslothai/unsloth.git to commit d91183d03feca2539a946354ca4fdbb4928e273c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 28.0 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0


### Step 2
Import unlosth and download qwen-2.5 3B model

In [6]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit",
    load_in_4bit=True,
)


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


### Inference with the base Qwen model

In [9]:
FastModel.for_inference(model)
messages = [
    {"role": "user", "content": "Explain what LoRA is in simple terms."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
)
print(outputs)


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,    840,  20772,   1128,
           6485,   5609,    374,    304,   4285,   3793,     13, 151645,    198,
         151644,  77091,    198,   4262,   5609,  13352,    369,  83267,  85896,
          28807,  57739,     13,   1084,    594,    264,   1714,   1483,    311,
          10515,    323,   5452,    855,  68924,   4119,    803,  29720,     11,
           7945,   5390,    369,   9155,   4119,    476,    979,    498,   1366,
            311,    912,   5107,   8654,    311,    264,   1614,   2041,  11941,
           7703,   1181,   1379,    382,    641,  34288,   3793,     11,  12793,
            498,    614,    264,   2409,   3745,    315,  23069,    320,     64,
           3460,   1614,    701,    714,    498,   1172,   1366,    311,   1486,
            448,   1045,    

In [10]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Explain what LoRA is in simple terms.
assistant
LoRA stands for Lightweight Recursive Adapter Optimization. It's a method used to adapt and scale pre-trained models more efficiently, particularly useful for smaller models or when you want to add additional capacity to a model without significantly increasing its size.

In simpler terms, imagine you have a big box of toys (a large model), but you only want to play with some of them (selectively fine-tune the model). LoRA allows you to do this in a way that uses less space (memory) than other methods, making it easier to work with larger models or more complex tasks on devices with limited resources.


### GPU memory cell

In [11]:
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print(f"GPU Memory : {gpu_memory:.2f} GB")
print(f"Allocated : {allocated:.2f} GB")
print(f"Reserved : {reserved:.2f} GB")

GPU Memory : 14.56 GB
Allocated : 2.21 GB
Reserved : 7.97 GB


## Trainable parameters

In [12]:
def print_trainable_parameters(model):

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print(f"Total Parameters     : {total:,}")
    print(f"Trainable Parameters : {trainable:,}")